# Self-Hosted Conversational SDR Voice Agent (Colab Prototype)

Fully self-hosted STT (Whisper) + LLM (Qwen2.5 via Ollama) + TTS (OmniVoice)
voice pipeline on LiveKit Agents, exposed via a self-hosted `livekit-server`
tunneled through ngrok. Test by connecting through LiveKit's Agents Playground.

Non-goals for this notebook: telephony, CRM, custom UI, compliance handling.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU in Colab"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
%pip install -q \
    livekit-agents~=1.0 \
    "livekit-plugins-openai~=1.0" \
    "livekit-plugins-silero~=1.0" \
    faster-whisper \
    ollama \
    pyngrok \
    soundfile

# OmniVoice: try PyPI first, fall back to installing from GitHub source
%pip install -q omnivoice || %pip install -q "git+https://github.com/k2-fsa/OmniVoice.git"

In [ ]:
import livekit.agents
import livekit.plugins.openai
import livekit.plugins.silero
import faster_whisper
import omnivoice
import ollama
import pyngrok
print("All imports OK")

In [ ]:
!curl -sSL https://get.livekit.io | bash
!livekit-server --version

In [ ]:
import urllib.request

REFERENCE_AUDIO_PATH = "jfk.flac"
REFERENCE_TRANSCRIPT_SNIPPET = "ask not what your country can do for you"

urllib.request.urlretrieve(
    "https://github.com/openai/whisper/raw/main/tests/jfk.flac",
    REFERENCE_AUDIO_PATH,
)
print("Downloaded:", REFERENCE_AUDIO_PATH)

In [ ]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
print("Whisper model loaded")

In [ ]:
segments, info = whisper_model.transcribe(REFERENCE_AUDIO_PATH, language="en")
transcript = " ".join(segment.text for segment in segments).strip()
print("Transcript:", transcript)
assert REFERENCE_TRANSCRIPT_SNIPPET in transcript.lower(), f"Expected snippet not found in: {transcript}"
print("STT verification PASSED")

In [ ]:
import subprocess
import time

!curl -fsSL https://ollama.com/install.sh | sh

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
print("Ollama server starting, PID:", ollama_process.pid)

In [ ]:
!ollama pull qwen2.5:7b-instruct

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "qwen2.5:7b-instruct",
        "messages": [{"role": "user", "content": "Reply with exactly the word: PONG"}],
        "max_tokens": 10,
    },
    timeout=60,
)
response.raise_for_status()
reply = response.json()["choices"][0]["message"]["content"]
print("LLM reply:", reply)
assert "PONG" in reply.upper(), f"Unexpected reply: {reply}"
print("LLM verification PASSED")

In [ ]:
import torch
from omnivoice import OmniVoice

omnivoice_model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)
print("OmniVoice model loaded")

In [ ]:
import numpy as np
import soundfile as sf

TEST_SENTENCE = "Hi, this is a quick test of the voice pipeline."

audio_chunks = omnivoice_model.generate(
    text=TEST_SENTENCE,
    instruct="female, medium pitch, american accent, friendly sales tone",
)
audio = audio_chunks[0]

assert isinstance(audio, np.ndarray), f"Expected np.ndarray, got {type(audio)}"
assert audio.ndim == 1 and audio.shape[0] > 0, f"Unexpected shape: {audio.shape}"

duration_seconds = audio.shape[0] / 24000
rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
print(f"Duration: {duration_seconds:.2f}s, RMS: {rms:.4f}")
assert duration_seconds > 0.5, "Audio too short — synthesis likely failed"
assert rms > 0.001, "Audio is near-silent — synthesis likely failed"

sf.write("tts_test_output.wav", audio, 24000)
print("TTS verification PASSED — listen to tts_test_output.wav to confirm quality")

In [ ]:
from IPython.display import Audio, display
display(Audio("tts_test_output.wav"))

Before running the next cell, sign up free at https://dashboard.ngrok.com/signup,
copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken,
and paste it below. Also reserve one static TCP address at
https://dashboard.ngrok.com/cloud-edge/tcp-addresses (free tier includes one) —
note its host and port, you'll need them in Step 3.

In [ ]:
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"  # from dashboard.ngrok.com
conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("ngrok authenticated")

Set `RESERVED_TCP_PORT` below to the port number from the static TCP address
you reserved in Step 1 (the part after the colon, e.g. for
`6.tcp.ngrok.io:19302` the port is `19302`). This makes `livekit-server`
bind its media/ICE port on the exact same port number ngrok forwards to,
which keeps the address it advertises to browsers correct.

In [ ]:
RESERVED_TCP_HOST = "PASTE_RESERVED_HOST_HERE"   # e.g. "6.tcp.ngrok.io"
RESERVED_TCP_PORT = 0                             # e.g. 19302

import socket
resolved_ip = socket.gethostbyname(RESERVED_TCP_HOST)
print(f"{RESERVED_TCP_HOST} resolves to {resolved_ip}")

config_yaml = f"""
port: 7880
rtc:
  tcp_port: {RESERVED_TCP_PORT}
  use_external_ip: false
  node_ip: {resolved_ip}
keys:
  devkey: secret
"""
with open("livekit-config.yaml", "w") as f:
    f.write(config_yaml)
print(config_yaml)

In [ ]:
import subprocess
import time

livekit_process = subprocess.Popen(
    ["livekit-server", "--config", "livekit-config.yaml"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
assert livekit_process.poll() is None, "livekit-server exited immediately — check config"
print("livekit-server running, PID:", livekit_process.pid)

In [ ]:
signaling_tunnel = ngrok.connect(7880, "tcp")
media_tunnel = ngrok.connect(
    RESERVED_TCP_PORT, "tcp", remote_addr=f"{RESERVED_TCP_HOST}:{RESERVED_TCP_PORT}"
)

print("Signaling tunnel:", signaling_tunnel.public_url)
print("Media tunnel:", media_tunnel.public_url)

# Convert the signaling tunnel's tcp:// URL into the ws:// form the LiveKit
# SDK/Playground expects.
LIVEKIT_WS_URL = signaling_tunnel.public_url.replace("tcp://", "ws://")
LIVEKIT_API_KEY = "devkey"
LIVEKIT_API_SECRET = "secret"
print("LIVEKIT_WS_URL:", LIVEKIT_WS_URL)

In [ ]:
import socket

host, port = LIVEKIT_WS_URL.replace("ws://", "").split(":")
with socket.create_connection((host, int(port)), timeout=10) as sock:
    print(f"Successfully connected to {host}:{port}")

In [ ]:
from livekit.agents import stt
from livekit.agents.utils import AudioBuffer
from livekit import rtc
import io

class WhisperSTT(stt.STT):
    def __init__(self):
        super().__init__(
            capabilities=stt.STTCapabilities(
                streaming=False,
                interim_results=False,
            )
        )

    async def _recognize_impl(self, buffer: AudioBuffer, *, language=None, conn_options=None) -> stt.SpeechEvent:
        wav_bytes = io.BytesIO(rtc.combine_audio_frames(buffer).to_wav_bytes())

        segments, _info = whisper_model.transcribe(wav_bytes, language="en")
        text = " ".join(segment.text for segment in segments).strip()

        # Empty or very short output means silence/noise rather than real
        # speech. Forward a marker instead of the raw (empty) text so the
        # LLM's system prompt can react by asking the caller to repeat,
        # rather than the LLM receiving nothing to respond to.
        if len(text) < 2:
            text = "[SILENCE_OR_UNCLEAR_AUDIO]"

        return stt.SpeechEvent(
            type=stt.SpeechEventType.FINAL_TRANSCRIPT,
            alternatives=[stt.SpeechData(language="en", text=text)],
        )

print("WhisperSTT plugin defined")

In [ ]:
import soundfile as sf_read

pcm_data, sample_rate = sf_read.read(REFERENCE_AUDIO_PATH, dtype="int16")
frame = rtc.AudioFrame(
    data=pcm_data.tobytes(),
    sample_rate=sample_rate,
    num_channels=1,
    samples_per_channel=len(pcm_data),
)

whisper_stt_plugin = WhisperSTT()
event = await whisper_stt_plugin._recognize_impl(buffer=[frame], language="en", conn_options=None)
plugin_transcript = event.alternatives[0].text
print("Plugin transcript:", plugin_transcript)
assert REFERENCE_TRANSCRIPT_SNIPPET in plugin_transcript.lower(), f"Expected snippet not found in: {plugin_transcript}"
print("STT plugin verification PASSED")

In [ ]:
from livekit.agents import tts
import numpy as np

SDR_VOICE_INSTRUCT = "female, medium pitch, american accent, friendly sales tone"

class OmniVoiceTTS(tts.TTS):
    def __init__(self):
        super().__init__(
            capabilities=tts.TTSCapabilities(streaming=False),
            sample_rate=24000,
            num_channels=1,
        )

    def synthesize(self, text: str, *, conn_options=None) -> "tts.ChunkedStream":
        return _OmniVoiceChunkedStream(tts=self, input_text=text, conn_options=conn_options)

class _OmniVoiceChunkedStream(tts.ChunkedStream):
    async def _run(self, output_emitter: "tts.AudioEmitter") -> None:
        audio_chunks = omnivoice_model.generate(
            text=self.input_text,
            instruct=SDR_VOICE_INSTRUCT,
        )
        audio = audio_chunks[0]
        pcm16 = (audio * 32767.0).astype(np.int16)

        output_emitter.initialize(
            request_id="",
            sample_rate=24000,
            num_channels=1,
            mime_type="audio/pcm",
        )
        output_emitter.push(pcm16.tobytes())
        output_emitter.flush()

print("OmniVoiceTTS plugin defined")

In [ ]:
omnivoice_tts_plugin = OmniVoiceTTS()
chunked_stream = omnivoice_tts_plugin.synthesize("This is a plugin verification test.")

collected_frames = []
async for synthesized_audio in chunked_stream:
    collected_frames.append(synthesized_audio.frame)

assert len(collected_frames) > 0, "No audio frames produced"
total_samples = sum(f.samples_per_channel for f in collected_frames)
duration_seconds = total_samples / 24000
print(f"Frames: {len(collected_frames)}, duration: {duration_seconds:.2f}s")
assert duration_seconds > 0.3, "Audio too short — plugin synthesis likely failed"
print("TTS plugin verification PASSED")

In [ ]:
SDR_INSTRUCTIONS = """
You are Alex, a friendly sales development rep for "Streamline", a fictional
project-management SaaS tool for small teams.

Your job on this call: briefly introduce yourself and Streamline, ask 1-2
discovery questions about how the prospect currently manages projects, and
gauge interest in a short demo. Keep every response to 1-3 sentences —
this is a voice conversation, not a chat, so avoid long monologues.

If the prospect raises an objection (too expensive, already using a tool,
not interested), acknowledge it briefly and ask one clarifying question
before moving on. If they clearly want to end the call, thank them for
their time and wrap up politely.

If you ever receive the exact message "[SILENCE_OR_UNCLEAR_AUDIO]", this
means the audio was silent or unintelligible — do not treat it as
something the prospect said. Simply and briefly ask them to repeat what
they said (e.g. "Sorry, I didn't catch that — could you say that again?").
""".strip()

print(SDR_INSTRUCTIONS)

In [ ]:
from livekit.agents import Agent, AgentSession
from livekit.plugins import openai, silero

def create_sdr_agent_session() -> AgentSession:
    return AgentSession(
        stt=WhisperSTT(),
        llm=openai.LLM.with_ollama(
            model="qwen2.5:7b-instruct",
            base_url="http://localhost:11434/v1",
        ),
        tts=OmniVoiceTTS(),
        vad=silero.VAD.load(),
    )

class SDRAgent(Agent):
    def __init__(self):
        super().__init__(instructions=SDR_INSTRUCTIONS)

print("AgentSession factory defined")